# 01 — Exploratory Data Analysis

Mobile Network Traffic Prediction — data loading, visual exploration, stationarity tests, decomposition, and ACF/PACF analysis.

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss

from src.data_loader import load_raw_data
from src.utils import load_config, save_fig

cfg = load_config('../config.yaml')
data_cfg = cfg['data']

## 1. Load data

In [ ]:
df = load_raw_data(
    '../' + data_cfg['raw_path'],
    data_cfg['datetime_column'],
    data_cfg['frequency'],
)
target = data_cfg['target_column']
df.head()

In [ ]:
summary = df[target].describe()
summary.to_csv('../results/tables/data_summary.csv')
summary

## 2. EDA visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
df[target].plot(ax=ax, title='Mobile Network Traffic Over Time')
save_fig(fig, '../results/plots/exploratory_analysis/traffic_over_time.png')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df[target].hist(bins=50, ax=ax)
ax.set_title('Distribution of Traffic Volume')
save_fig(fig, '../results/plots/exploratory_analysis/traffic_distribution.png')

## 3. Stationarity tests (ADF & KPSS)

In [ ]:
adf_result = adfuller(df[target].dropna())
kpss_result = kpss(df[target].dropna(), regression='c', nlags='auto')

print('ADF statistic:', adf_result[0], 'p-value:', adf_result[1])
print('KPSS statistic:', kpss_result[0], 'p-value:', kpss_result[1])

## 4. Time series decomposition

In [ ]:
decomposition = seasonal_decompose(df[target].dropna(), model='additive', period=24)
fig = decomposition.plot()
fig.set_size_inches(12, 8)
save_fig(fig, '../results/plots/time_series_decomposition/additive_decomposition.png')

## 5. ACF / PACF analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df[target].dropna(), ax=axes[0], lags=48)
plot_pacf(df[target].dropna(), ax=axes[1], lags=48)
save_fig(fig, '../results/plots/exploratory_analysis/acf_pacf.png')